In [0]:
from pyspark.sql.functions import col, lit

policy_schema= "policy_id int, policy_type string, customer_id int, start_date timestamp, end_date timestamp, premium double,  coverage_amount double"

df = spark.read \
    .schema(policy_schema) \
    .json("abfss://landing@projpolicysytem.dfs.core.windows.net/policy/")
df.display()

In [0]:

df_with_flag = df.withColumn("merge_flag", lit(False))

bronze_path = "abfss://bronzelayer@projpolicysytem.dfs.core.windows.net/policy"

df_with_flag.write \
    .format("delta") \
    .mode("append") \
    .option("path", bronze_path) \
    .saveAsTable("policyprojcatalog.policyprojdb.policy")

In [0]:
%sql
select * from policyprojcatalog.policyprojdb.policy

In [0]:
dbutils.fs.mv(
    "abfss://processed@projpolicysytem.dfs.core.windows.net/policy/",
    "abfss://landing@projpolicysytem.dfs.core.windows.net/PolicyData/",
    True
)

In [0]:
from datetime import datetime

current_time = datetime.now().strftime('%m-%d-%Y')

new_folder = f"abfss://processed@projpolicysytem.dfs.core.windows.net/PolicyData/{current_time}"

dbutils.fs.mv(
    "abfss://landing@projpolicysytem.dfs.core.windows.net/policy/",
    new_folder,
    True
)